In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

titles = []
prices = []
reviews = []

base_url = "https://www.amazon.in/s?k=iphone&crid=1ZHQN7M1NZNKE&sprefix=iphone+%2Caps%2C305&ref=nb_sb_noss_2"

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36 Avast/117.0.0.0',
    'Accept-Language': 'en-US, en;q=0.5'
}

num_pages = 100 

for page in range(1, num_pages + 1):

    url = base_url + "&page=" + str(page)

    response = requests.get(url, headers=HEADERS)

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        products = soup.find_all('div', class_='s-result-item')

        for product in products:
            title = product.find('span', class_='a-text-normal')
            price = product.find('span', class_='a-offscreen')
            review = product.find('span', {'aria-label': True})

            if title and price and review:
                titles.append(title.text)
                prices.append(price.text)
                reviews.append(review['aria-label'][0:3])

    else:
        print(f'Failed to retrieve data from page {page}')

data = {
    'Name': titles,
    'Price': prices,
    'Review': reviews
}
df = pd.DataFrame(data)


In [44]:
df['Review'] = df['Review'].str.replace('[^\d.]', '', regex=True)
df['Review'] = pd.to_numeric(df['Review'], errors='coerce')
df['Price'] = df['Price'].str.replace('₹','').str.replace(',','').astype(float)
df = df[df['Name'].str.startswith('Apple')]
df.reset_index(drop=True, inplace=True)
df

,Name,Price,Review
0,Apple iPhone 13 (128GB) - Midnight,49999.0,4.6
1,Apple iPhone 15 (128 GB) - Green,79900.0,5.0
2,Apple iPhone 13 (128GB) - Pink,49999.0,4.6
3,Apple iPhone 13 (128GB) - Blue,49999.0,4.6
4,Apple iPhone 13 (128GB) - Starlight,49999.0,4.6
5,Apple iPhone 13 (128GB) - Green,49999.0,4.6
6,Apple iPhone 13 (128GB) - Midnight,49999.0,4.6
7,Apple iPhone 15 Plus (128 GB) - Black,89900.0,4.0
8,Apple iPhone 15 (512 GB) - Blue,109900.0,NaN
9,Apple iPhone 13 (128GB) - (Product) RED,49999.0,4.6


In [45]:
df.to_csv('Apple_Amazone_Data.csv', index= False)